# SQL Data Modeling & Relational Analysis (Relational Foundation)

This notebook demonstrates how relational data from the Olist e-commerce dataset is structured, joined, and queried using SQL.

The goal is to:
- Understand relationships between entities (orders, customers, reviews, items)
- Build a unified analytical dataset at the order level
- Perform initial aggregations using SQL before moving to Python for further analysis and modeling

In [12]:
import pandas as pd
import sqlite3
from pathlib import Path

# Connect to database
conn = sqlite3.connect("../data/processed/olist.db")

In [13]:
query = """
SELECT name FROM sqlite_master WHERE type='table';
"""
pd.read_sql(query, conn)

,name
0,orders
1,customers
2,reviews
3,items
4,products
5,leads
6,deals


## Data Model Overview

The dataset consists of multiple related tables:

- **orders**: core transactional table (one row per order)
- **customers**: customer information (linked via customer_id)
- **reviews**: customer feedback (linked via order_id)
- **order_items**: product-level details per order
- **products**: product metadata

### Key Relationships:
- orders.customer_id → customers.customer_id
- orders.order_id → reviews.order_id
- orders.order_id → order_items.order_id
- order_items.product_id → products.product_id

### Important Note:
- customer_unique_id represents the real customer across multiple orders

In [14]:
# Preview key tables
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [15]:
pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:
pd.read_sql("SELECT * FROM reviews LIMIT 5;", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [17]:
# Core JOIN (build analytical base)
query = """
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    r.review_comment_title,
    r.review_comment_message
FROM orders o
LEFT JOIN customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN reviews r 
    ON o.order_id = r.order_id
"""
df_sql_base = pd.read_sql(query, conn)

df_sql_base.head()

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_title,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,4.0,None,"Não testei o produto ainda, mas ele veio corre..."
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,4.0,Muito boa a loja,Muito bom o produto.
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,5.0,None,None
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,5.0,None,O produto foi exatamente o que eu esperava e e...
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,5.0,None,None


## Analytical Grain

The dataset is structured at the **order level**:
- Each row represents one completed order
- Includes customer identity, delivery timestamps, and review outcome

This structure allows analysis of customer experience at the transaction level.

In [18]:
# Order value aggregation
query = """
SELECT
    order_id,
    SUM(price + freight_value) AS order_value,
    COUNT(product_id) AS product_count
FROM items
GROUP BY order_id
"""
order_value_df = pd.read_sql(query, conn)

order_value_df.head()

,order_id,order_value,product_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1


In [19]:
# Join order value to base
df_sql_full = df_sql_base.merge(order_value_df, on="order_id", how="left")

df_sql_full.head()

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_title,review_comment_message,order_value,product_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,4.0,None,"Não testei o produto ainda, mas ele veio corre...",38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,4.0,Muito boa a loja,Muito bom o produto.,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,5.0,None,None,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,5.0,None,O produto foi exatamente o que eu esperava e e...,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,5.0,None,None,28.62,1.0


In [20]:
# SQL aggregation example: dissatisfaction rate by month
query = """
SELECT
    strftime('%m', o.order_purchase_timestamp) AS month,
    AVG(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) AS dissatisfaction_rate
FROM orders o
LEFT JOIN reviews r
    ON o.order_id = r.order_id
GROUP BY month
ORDER BY month;
"""
pd.read_sql(query, conn)

,month,dissatisfaction_rate
0,01,0.153818
1,02,0.196698
2,03,0.203076
3,04,0.135135
4,05,0.121292
5,06,0.113757
6,07,0.114910
7,08,0.110672
8,09,0.126299
9,10,0.141570


In [21]:
# Customer behavior via SQL
query = """
SELECT
    customer_unique_id,
    COUNT(order_id) AS order_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
HAVING order_count > 1
LIMIT 10;
"""
pd.read_sql(query, conn)

,customer_unique_id,order_count
0,00172711b30d52eea8b313a7f2cced02,2
1,004288347e5e88a27ded2bb23747066c,2
2,004b45ec5c64187465168251cd1c9c2f,2
3,0058f300f57d7b93c477a131a59b36c3,2
4,00a39521eb40f7012db50455bf083460,2
5,00cc12a6d8b578b8ebd21ea4e2ae8b27,2
6,011575986092c30523ecb71ff10cb473,2
7,011b4adcd54683b480c4d841250a987f,2
8,012452d40dafae4df401bced74cdb490,2
9,012a218df8995d3ec3bb221828360c86,2


## Key SQL Insights

- Dissatisfaction can already be observed at the SQL level through aggregation
- Order value and product count enrich the understanding of customer experience
- Customer-level aggregation reveals repeat purchase behavior

SQL is used here to:
- Structure relational data
- Create analytical datasets
- Perform initial aggregations

Further feature engineering and modeling will be performed in Python.

In [22]:
# Save dataset for pipeline
# df_sql_full.to_csv("../data/processed/main_dataset.csv", index=False)